# Amazon ML Challenge 2026: Stage 3 (Cross-Encoder Reranking) & Stage 4 (Cluster Consistency)

**Input:** `score_C_train.parquet` from NB 03, normalized parquets from NB 00

**Pipeline position:** These are **refinement stages** between the LightGBM scorer and the
Decision Layer. Stage 3 selectively reranks ambiguous pairs with a cross-encoder. Stage 4
checks cluster consistency and applies lightweight conflict rules.

**Methodology reference:**
- Stage 3: D1–D6 (Ditto-style cross-encoder), 2.5 (ambiguous margin band)
- Stage 4: E1–E6 (cluster-level matching), E5 (consistency checks)

**Outputs:**
- `score_refined_train.parquet` — Refined scores with cross-encoder blending + consistency flags
- Stage 4 agreement report (determines if full clustering is needed)

**Requirements:** GPU notebook (T4) for cross-encoder. If band is too large, cross-encoder is skipped.

In [ ]:
import os, sys, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# PATH RESOLUTION
# ============================================================

IS_KAGGLE = os.path.exists('/kaggle')

def find_dir_by_file(pattern):
    for p in Path('/kaggle/input').rglob(pattern):
        return p.parent
    return None

if IS_KAGGLE:
    WORK_DIR = Path('/kaggle/working')
    SCORE_DIR = find_dir_by_file('score_C_train.parquet')
    if SCORE_DIR is None:
        SCORE_DIR = WORK_DIR
    NORM_DIR = find_dir_by_file('train_s1_norm.parquet')
    if NORM_DIR is None:
        NORM_DIR = Path('/kaggle/input/notebooks/ojassangwai/ml1nbv1/normalized')
    DATA_DIR = find_dir_by_file('train_ground_truth.tsv')
    if DATA_DIR is None:
        DATA_DIR = Path('/kaggle/input/datasets/ojassangwai/mlchallengedata')
else:
    WORK_DIR = Path('work')
    SCORE_DIR = WORK_DIR
    NORM_DIR = WORK_DIR / 'normalized'
    DATA_DIR = Path('data/student_resource/dataset')

OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Score Dir: {SCORE_DIR}")
print(f"Norm Dir:  {NORM_DIR}")
print(f"Data Dir:  {DATA_DIR}")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device:    {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU:       {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading OOF scores from NB 03...")
df_scores = pd.read_parquet(SCORE_DIR / 'score_C_train.parquet')
print(f"  Loaded {len(df_scores):,} scored pairs")

# Ground truth
print("Loading ground truth...")
df_gt = pd.read_csv(DATA_DIR / 'train_ground_truth.tsv', sep='\t', dtype=str)
gt_map = {}
for sid, raw_m in zip(
    df_gt['source1_entity_id'].astype(str).str.strip(),
    df_gt['matched_entity_ids'].fillna('').astype(str)
):
    if raw_m and raw_m != 'nan':
        gt_map[sid] = {m.strip() for m in raw_m.split(',') if m.strip()}
    else:
        gt_map[sid] = set()
del df_gt
gc.collect()
print(f"  GT entities: {len(gt_map):,}")

In [ ]:
# ============================================================
# STAGE 3: AMBIGUOUS BAND AUDIT (Methodology 2.5, D5)
# Measures how many pairs fall in [threshold ± margin].
# If the band is small enough for GPU budget, proceed with reranking.
# ============================================================

THRESHOLD = 0.75
MARGIN = 0.10
MAX_GPU_HOURS = 3.0     # Budget: max hours on T4 for cross-encoder
CE_PAIRS_PER_SEC = 1500  # Conservative estimate for T4 fp16

scores = df_scores['score_C'].values
low_bound = max(0.0, THRESHOLD - MARGIN)
high_bound = min(1.0, THRESHOLD + MARGIN)

ambiguous_mask = (scores >= low_bound) & (scores <= high_bound)
band_size = int(np.sum(ambiguous_mask))
band_pct = (band_size / len(scores) * 100) if len(scores) > 0 else 0.0
est_gpu_hours = (band_size / CE_PAIRS_PER_SEC) / 3600
feasible = est_gpu_hours <= MAX_GPU_HOURS

print("\n" + "=" * 60)
print(" STAGE 3: AMBIGUOUS BAND AUDIT")
print("=" * 60)
print(f" Threshold: {THRESHOLD:.3f} | Margin: ±{MARGIN:.3f} ([{low_bound:.3f}, {high_bound:.3f}])")
print(f" Total Candidate Pairs:    {len(scores):,}")
print(f" Ambiguous Band Pairs:     {band_size:,} ({band_pct:.2f}%)")
print(f" Est. T4 GPU Inference:     {est_gpu_hours:.2f} hours")
print(f" Budget:                    {MAX_GPU_HOURS:.1f} hours")
print(f" Decision:                  {'✅ PROCEED WITH RERANKING' if feasible else '⏭️ SKIP RERANKING (too expensive)'}")
print("=" * 60)

# Score distribution around the decision boundary
print(f"\n  Score distribution near threshold:")
for lo, hi, label in [
    (0.0, 0.1, '0.0-0.1'), (0.1, 0.3, '0.1-0.3'), (0.3, 0.5, '0.3-0.5'),
    (0.5, 0.6, '0.5-0.6'), (0.6, 0.7, '0.6-0.7'), (0.7, 0.8, '0.7-0.8'),
    (0.8, 0.9, '0.8-0.9'), (0.9, 1.01, '0.9-1.0'),
]:
    count = int(np.sum((scores >= lo) & (scores < hi)))
    print(f"    {label}: {count:>12,}")

In [ ]:
# ============================================================
# STAGE 3: CROSS-ENCODER RERANKING (Methodology D1-D3)
# Only runs if the ambiguous band is within GPU budget.
# Uses cross-encoder/ms-marco-MiniLM-L-6-v2 (MIT license, 22M params).
# Serialization: Ditto-style "name | address" pairs.
# Blends: final_score = alpha * score_C + (1-alpha) * score_D
# ============================================================

CE_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
CE_BATCH_SIZE = 256 if DEVICE == 'cuda' else 32
CE_MAX_LENGTH = 128
BLEND_ALPHA = 0.5   # Equal weight to GBDT and cross-encoder

df_refined = df_scores.copy()
df_refined['reranked_by_D'] = 0

if feasible and band_size > 0:
    print(f"\nLoading cross-encoder: {CE_MODEL_NAME}")
    ce_tokenizer = AutoTokenizer.from_pretrained(CE_MODEL_NAME)
    ce_model = AutoModelForSequenceClassification.from_pretrained(CE_MODEL_NAME).to(DEVICE)
    ce_model.eval()
    
    # Load normalized records for text serialization
    print("Loading normalized records...")
    df_s1 = pd.read_parquet(NORM_DIR / 'train_s1_norm.parquet')
    df_s2 = pd.read_parquet(NORM_DIR / 'train_s2_norm.parquet')
    df_s3 = pd.read_parquet(NORM_DIR / 'train_s3_norm.parquet')
    df_tgt = pd.concat([df_s2, df_s3], ignore_index=True)
    del df_s2, df_s3
    gc.collect()
    
    s1_dict = {r['entity_id']: r for _, r in df_s1.iterrows()}
    tgt_dict = {r['entity_id']: r for _, r in df_tgt.iterrows()}
    del df_s1, df_tgt
    gc.collect()
    
    # Get ambiguous pair indices
    ambiguous_indices = np.where(ambiguous_mask)[0]
    s1_ids = df_scores['s1_id'].values
    cand_ids = df_scores['candidate_id'].values
    
    # Build text pairs (Ditto-style serialization: "name | address")
    print(f"  Serializing {len(ambiguous_indices):,} pairs...")
    pairs_text_a = []
    pairs_text_b = []
    for idx in ambiguous_indices:
        sid = s1_ids[idx]
        cid = cand_ids[idx]
        s1_rec = s1_dict.get(sid, {})
        t_rec = tgt_dict.get(cid, {})
        text_a = f"{s1_rec.get('name_core', '')} | {s1_rec.get('addr_core', '')}"
        text_b = f"{t_rec.get('name_core', '')} | {t_rec.get('addr_core', '')}"
        pairs_text_a.append(text_a)
        pairs_text_b.append(text_b)
    
    # Cross-encoder inference in batches
    print(f"  Running cross-encoder inference ({len(pairs_text_a):,} pairs, batch_size={CE_BATCH_SIZE})...")
    ce_scores = []
    t_ce = time.time()
    
    with torch.no_grad():
        for i in range(0, len(pairs_text_a), CE_BATCH_SIZE):
            batch_a = pairs_text_a[i:i+CE_BATCH_SIZE]
            batch_b = pairs_text_b[i:i+CE_BATCH_SIZE]
            
            inputs = ce_tokenizer(
                batch_a, batch_b,
                padding=True, truncation=True,
                max_length=CE_MAX_LENGTH,
                return_tensors='pt'
            ).to(DEVICE)
            
            with torch.amp.autocast(device_type=DEVICE, enabled=(DEVICE=='cuda')):
                logits = ce_model(**inputs).logits
            
            if logits.shape[1] == 1:
                probs = torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
            else:
                probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            ce_scores.extend(probs)
            
            if (i // CE_BATCH_SIZE) % 100 == 0 and i > 0:
                elapsed = time.time() - t_ce
                pairs_per_sec = i / elapsed
                print(f"    {i:,} / {len(pairs_text_a):,} ({pairs_per_sec:.0f} pairs/s)")
    
    ce_scores = np.array(ce_scores, dtype=np.float32)
    elapsed_ce = time.time() - t_ce
    print(f"  Cross-encoder done in {elapsed_ce:.1f}s ({len(ce_scores)/elapsed_ce:.0f} pairs/s)")
    
    # Blend scores: weighted average
    original_scores = df_refined['score_C'].values.copy()
    blended = BLEND_ALPHA * original_scores[ambiguous_indices] + (1 - BLEND_ALPHA) * ce_scores
    df_refined.loc[df_refined.index[ambiguous_indices], 'score_C'] = blended
    df_refined.loc[df_refined.index[ambiguous_indices], 'reranked_by_D'] = 1
    
    # Report movement
    moved_up = np.sum(blended > original_scores[ambiguous_indices])
    moved_down = np.sum(blended < original_scores[ambiguous_indices])
    print(f"\n  Blending result (alpha={BLEND_ALPHA}):")
    print(f"    Moved UP:   {moved_up:,} pairs")
    print(f"    Moved DOWN: {moved_down:,} pairs")
    print(f"    Unchanged:  {len(ce_scores) - moved_up - moved_down:,} pairs")
    
    del ce_model, ce_tokenizer, pairs_text_a, pairs_text_b, ce_scores
    del s1_dict, tgt_dict
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

else:
    print("\n  Cross-encoder reranking SKIPPED (band too large or empty).")
    print("  Passing scores through unchanged.")

In [ ]:
# ============================================================
# STAGE 4: CLUSTER CONSISTENCY PRE-CHECK (Methodology E1-E6)
# Compares independent pairwise thresholding against a
# cluster-consistent decision. If agreement >= 97%, skip
# full graph clustering and use lightweight conflict rules.
# ============================================================

print("\n" + "=" * 60)
print(" STAGE 4: CLUSTER CONSISTENCY PRE-CHECK")
print("=" * 60)

# 1. Independent thresholding
refined_scores = df_refined['score_C'].values
s1_ids_arr = df_refined['s1_id'].values
cand_ids_arr = df_refined['candidate_id'].values

pred_indep = {}  # s1_id -> set of predicted candidate IDs
for i in range(len(df_refined)):
    sid = s1_ids_arr[i]
    if refined_scores[i] >= THRESHOLD:
        pred_indep.setdefault(sid, set()).add(cand_ids_arr[i])
    elif sid not in pred_indep:
        pred_indep[sid] = set()

# 2. Build record -> ground truth entity mapping
# If r in M(e), then record_to_entity[r] = e
record_to_entity = {}
for sid, true_matches in gt_map.items():
    for mid in true_matches:
        record_to_entity[mid] = sid

# 3. Check cluster consistency:
# For each entity, check if its predicted records all belong
# to the same ground-truth cluster (or are consistent).
agreements = 0
disagreements_detail = []  # Sample disagreements for analysis
total_entities = len(gt_map)

for sid in gt_map:
    indep = pred_indep.get(sid, set())
    if not indep:
        agreements += 1
        continue
    
    # Check if all predicted records actually belong to this entity in GT
    all_correct = True
    for cid in indep:
        true_owner = record_to_entity.get(cid)
        if true_owner is not None and true_owner != sid:
            all_correct = False
            if len(disagreements_detail) < 20:
                disagreements_detail.append((sid, cid, true_owner))
            break
    if all_correct:
        agreements += 1

agreement_rate = agreements / total_entities if total_entities > 0 else 1.0
skip_full_clustering = agreement_rate >= 0.97

print(f" Entity Agreement Rate: {agreement_rate * 100:.2f}%")
print(f" Agreements:            {agreements:,} / {total_entities:,}")
if skip_full_clustering:
    print(" [DECISION] Agreement >= 97%: Skip full graph clustering.")
    print("            Use lightweight conflict-resolution rule instead.")
else:
    print(" [DECISION] Agreement < 97%: Full graph clustering may be warranted.")
print("=" * 60)

if disagreements_detail:
    print(f"\n  Sample cross-entity conflicts (first {len(disagreements_detail)}):")
    for sid, cid, true_owner in disagreements_detail[:5]:
        print(f"    Entity {sid} claimed record {cid}, but GT says it belongs to {true_owner}")

In [ ]:
# ============================================================
# STAGE 4: LIGHTWEIGHT CONFLICT RESOLUTION (Methodology E5)
# When multiple candidates pass threshold for an entity:
# - If address conflict (different house number or state),
#   drop the weaker candidate if score margin is significant.
# - If consistent addresses, keep both.
# ============================================================

print("\n" + "=" * 60)
print(" STAGE 4: LIGHTWEIGHT CONFLICT RESOLUTION")
print("=" * 60)

# Load target records for address conflict check
print("  Loading target records for conflict check...")
df_s2 = pd.read_parquet(NORM_DIR / 'train_s2_norm.parquet')
df_s3 = pd.read_parquet(NORM_DIR / 'train_s3_norm.parquet')
df_tgt = pd.concat([df_s2, df_s3], ignore_index=True)
del df_s2, df_s3

tgt_records = {}
for _, r in df_tgt.iterrows():
    tgt_records[r['entity_id']] = dict(r)
del df_tgt
gc.collect()

# Build per-entity candidate lists with scores (above threshold)
candidate_predictions = {}  # s1_id -> [(cand_id, score), ...]
for i in range(len(df_refined)):
    sid = s1_ids_arr[i]
    if refined_scores[i] >= THRESHOLD:
        candidate_predictions.setdefault(sid, []).append((cand_ids_arr[i], refined_scores[i]))

CONFLICT_MARGIN = 0.15
total_dropped = 0
total_kept = 0
total_entities_with_conflicts = 0

final_preds = {}  # s1_id -> set of final predicted candidate IDs

for sid in gt_map:
    cand_list = candidate_predictions.get(sid, [])
    
    if not cand_list:
        final_preds[sid] = set()
        continue
    
    if len(cand_list) == 1:
        final_preds[sid] = {cand_list[0][0]}
        total_kept += 1
        continue
    
    # Sort descending by score
    sorted_cands = sorted(cand_list, key=lambda x: -x[1])
    top_cid, top_score = sorted_cands[0]
    top_rec = tgt_records.get(top_cid, {})
    
    accepted = {top_cid}
    entity_had_conflict = False
    
    for cid, score in sorted_cands[1:]:
        rec = tgt_records.get(cid, {})
        
        # Check address conflict
        h_top = top_rec.get('house_number', '')
        h_cur = rec.get('house_number', '')
        h_conflict = (h_top and h_cur and h_top != h_cur)
        
        s_top = top_rec.get('state_code', '')
        s_cur = rec.get('state_code', '')
        s_conflict = (s_top and s_cur and s_top != s_cur)
        
        if (h_conflict or s_conflict) and (top_score - score >= CONFLICT_MARGIN):
            total_dropped += 1
            entity_had_conflict = True
            continue
        
        accepted.add(cid)
        total_kept += 1
    
    if entity_had_conflict:
        total_entities_with_conflicts += 1
    final_preds[sid] = accepted

del tgt_records
gc.collect()

print(f"  Entities with address conflicts: {total_entities_with_conflicts:,}")
print(f"  Candidates dropped by rule:      {total_dropped:,}")
print(f"  Candidates kept:                 {total_kept:,}")
print("=" * 60)

In [ ]:
# ============================================================
# SAVE REFINED SCORES
# ============================================================

print("\n" + "=" * 50)
print(" SAVING REFINED SCORES")
print("=" * 50)

# Add a consistency flag
df_refined['conflict_dropped'] = 0
# Mark rows that were dropped by conflict resolution
for i in range(len(df_refined)):
    sid = s1_ids_arr[i]
    cid = cand_ids_arr[i]
    if refined_scores[i] >= THRESHOLD:
        kept = final_preds.get(sid, set())
        if cid not in kept:
            df_refined.iloc[i, df_refined.columns.get_loc('conflict_dropped')] = 1

refined_path = WORK_DIR / 'score_refined_train.parquet'
df_refined.to_parquet(refined_path, index=False)
print(f"  Saved: {refined_path} ({refined_path.stat().st_size / 1e6:.1f} MB)")

# Quick F0.5 check after refinement
def compute_entity_f05(pred_ids, true_ids):
    if len(true_ids) == 0:
        return 1.0 if len(pred_ids) == 0 else 0.0
    if len(pred_ids) == 0:
        return 0.0
    tp = len(pred_ids & true_ids)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_ids)
    recall = tp / len(true_ids)
    denom = 0.25 * precision + recall
    return (1.25 * precision * recall) / denom if denom > 0 else 0.0

entity_scores = []
for sid in gt_map:
    pred = final_preds.get(sid, set())
    entity_scores.append(compute_entity_f05(pred, gt_map[sid]))

f05_after = np.mean(entity_scores)
print(f"\n  Macro F0.5 after Stage 3+4: {f05_after:.4f} (at threshold={THRESHOLD})")
print(f"  (Full two-threshold optimization in NB 05)")